# Enterprise Project 1 - Real-Time Supply Chain Control Tower Copilot

This notebook builds an enterprise-grade Foundry Agent pattern for a supply-chain control tower. It combines Azure AI Foundry Agents, OpenAPI API tools, a Copilot-style orchestration loop, and a simulated real-time event stream.

## Business scenario

A logistics team wants a Copilot that can triage shipment delays, inspect current weather risk, call enterprise operations APIs, generate an incident summary, and recommend next-best actions for a human coordinator.

## Architecture

1. Copilot surface: this notebook acts as the analyst-facing Copilot shell.
2. Foundry agent: a prompt agent receives operational events and decides which tools to use.
3. API tools: the agent can call public weather data and optional enterprise APIs behind your gateway.
4. Real-time stream: simulated shipment events show how this can plug into Event Hubs, Service Bus, Kafka, or a webhook feed.
5. Governance hooks: every response is captured with correlation IDs, severity, and recommended human approval points.

> Production note: replace `ENTERPRISE_API_BASE_URL` with a public HTTPS API endpoint, ideally Azure API Management or Azure Container Apps. Foundry-hosted tool calls cannot reach your notebook `localhost`.


## 1. Install Required Packages

The versions match the rest of this training repo where possible. Run this once per environment, then restart the kernel if Jupyter asks you to.


In [2]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity jsonref pandas rich


Note: you may need to restart the kernel to use updated packages.


## 2. Load Configuration

Create or reuse a `.env` file with these values:

```text
FOUNDRY_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com/api/projects/<project-name>
MODEL_DEPLOYMENT_NAME=<your-model-deployment>
ENTERPRISE_API_BASE_URL=https://<optional-enterprise-api-host>
```

`ENTERPRISE_API_BASE_URL` is optional. When it is not set, the notebook still creates a useful public weather-enabled agent and uses local sample business context in the prompt.


In [3]:
import os
import json
import time
import uuid
from datetime import datetime, timezone
from typing import Any

import jsonref
import pandas as pd
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv()
console = Console()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
enterprise_api_base_url = os.getenv("ENTERPRISE_API_BASE_URL", "").rstrip("/")

missing = [
    name for name, value in {
        "FOUNDRY_PROJECT_ENDPOINT": foundry_project_endpoint,
        "MODEL_DEPLOYMENT_NAME": model_deployment_name,
    }.items()
    if not value
]

if missing:
    raise ValueError(f"Missing required environment variables: {', '.join(missing)}")

console.print(Panel.fit("Configuration loaded", title="Supply Chain Copilot"))
print(f"Project endpoint: {foundry_project_endpoint}")
print(f"Model deployment: {model_deployment_name}")
print(f"Enterprise API configured: {bool(enterprise_api_base_url)}")


╭─ Supply Chain Copilot ─╮
│ Configuration loaded   │
╰────────────────────────╯

Project endpoint: https://sandeepfoundry2222.services.ai.azure.com/api/projects/sandeepfoundryproj2
Model deployment: grok-4.3
Enterprise API configured: False


## 3. Connect to Azure AI Foundry

`DefaultAzureCredential` supports Azure CLI login, managed identity, Visual Studio Code, and other Azure developer credentials.


In [4]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

openai_client = project_client.get_openai_client()
print("Foundry project client and OpenAI-compatible client are ready.")


Foundry project client and OpenAI-compatible client are ready.


## 4. Define OpenAPI Tool Specifications

The weather tool uses `wttr.in`, which is reachable publicly. The enterprise operations tool is optional and expects your own API gateway to implement inventory, shipment, and supplier-risk endpoints.


In [5]:
weather_openapi_spec: dict[str, Any] = {
    "openapi": "3.1.0",
    "info": {
        "title": "Weather Risk API",
        "description": "Retrieves current weather data for a location using wttr.in.",
        "version": "1.0.0",
    },
    "servers": [{"url": "https://wttr.in"}],
    "paths": {
        "/{location}": {
            "get": {
                "operationId": "GetCurrentWeather",
                "description": "Get weather information for a location.",
                "parameters": [
                    {"name": "location", "in": "path", "required": True, "description": "City, port, warehouse, or airport location.", "schema": {"type": "string"}},
                    {"name": "format", "in": "query", "required": True, "description": "Always use j1 for JSON output.", "schema": {"type": "string", "default": "j1"}},
                ],
                "responses": {"200": {"description": "Weather data returned successfully.", "content": {"application/json": {"schema": {"type": "object"}}}}},
            }
        }
    },
}

enterprise_ops_openapi_spec: dict[str, Any] | None = None
if enterprise_api_base_url:
    enterprise_ops_openapi_spec = {
        "openapi": "3.1.0",
        "info": {"title": "Enterprise Supply Chain Operations API", "description": "Operational API for inventory, shipment ETA, and supplier risk lookups.", "version": "1.0.0"},
        "servers": [{"url": enterprise_api_base_url}],
        "paths": {
            "/inventory/{sku}/status": {"get": {"operationId": "GetInventoryStatus", "description": "Get inventory availability and safety-stock risk for a SKU.", "parameters": [{"name": "sku", "in": "path", "required": True, "schema": {"type": "string"}}], "responses": {"200": {"description": "Inventory status", "content": {"application/json": {"schema": {"type": "object"}}}}}}},
            "/shipments/{order_id}/eta": {"get": {"operationId": "GetShipmentEta", "description": "Get shipment ETA and carrier exception details.", "parameters": [{"name": "order_id", "in": "path", "required": True, "schema": {"type": "string"}}], "responses": {"200": {"description": "Shipment ETA", "content": {"application/json": {"schema": {"type": "object"}}}}}}},
            "/suppliers/{supplier_id}/risk": {"get": {"operationId": "GetSupplierRisk", "description": "Get supplier risk posture, recent incidents, and mitigation guidance.", "parameters": [{"name": "supplier_id", "in": "path", "required": True, "schema": {"type": "string"}}], "responses": {"200": {"description": "Supplier risk", "content": {"application/json": {"schema": {"type": "object"}}}}}}},
        },
    }

weather_tool = {
    "type": "openapi",
    "openapi": {"name": "weather_risk", "spec": jsonref.loads(json.dumps(weather_openapi_spec)), "auth": {"type": "anonymous"}},
}

tools = [weather_tool]
if enterprise_ops_openapi_spec:
    tools.append({"type": "openapi", "openapi": {"name": "enterprise_supply_chain_ops", "spec": jsonref.loads(json.dumps(enterprise_ops_openapi_spec)), "auth": {"type": "anonymous"}}})

print(f"Registered {len(tools)} OpenAPI tool definition(s) for this agent.")


Registered 1 OpenAPI tool definition(s) for this agent.


## 5. Create the Supply Chain Agent

The instructions make the agent behave like an enterprise operations Copilot: concise, action-oriented, tool-aware, and explicit about uncertainty.


In [6]:
agent_name = "ent-supply-chain-control-tower-agent"

agent_instructions = """
You are an enterprise supply-chain control tower Copilot.

Your job:
- Triage shipment, supplier, inventory, and weather-risk events.
- Use available API tools whenever external facts are needed.
- Separate observed facts from assumptions.
- Recommend concrete actions for operations teams.
- Flag decisions requiring human approval.
- Keep responses structured for incident management systems.

Response format:
1. Executive summary
2. Severity and confidence
3. Evidence used
4. Recommended actions
5. Human approval required
6. Follow-up questions, only if needed
""".strip()

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(model=model_deployment_name, instructions=agent_instructions, tools=tools),
)

print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")


Agent created (id: ent-supply-chain-control-tower-agent:1, name: ent-supply-chain-control-tower-agent, version: 1)


## 6. Simulate a Real-Time Operations Stream

In production, replace this generator with Azure Event Hubs, Service Bus, Kafka, Azure Functions, or a webhook listener. The event envelope is intentionally close to what a real enterprise event contract would look like.


In [7]:
def build_event(event_type: str, payload: dict[str, Any]) -> dict[str, Any]:
    return {"event_id": str(uuid.uuid4()), "event_type": event_type, "source": "supply-chain-control-tower-demo", "occurred_at": datetime.now(timezone.utc).isoformat(), "payload": payload}

realtime_events = [
    build_event("shipment_delay", {"order_id": "SO-104488", "carrier": "Contoso Freight", "lane": "Newark Port to Chicago DC", "current_location": "New York City", "customer_tier": "platinum", "sku": "MED-PUMP-42", "supplier_id": "SUP-8842", "reported_delay_hours": 18}),
    build_event("inventory_risk", {"sku": "BATTERY-LFP-9K", "warehouse": "Dallas DC", "days_of_cover": 2.4, "demand_signal": "promotion spike", "current_location": "Dallas"}),
]

pd.DataFrame([event["payload"] | {"event_type": event["event_type"], "event_id": event["event_id"]} for event in realtime_events])


,order_id,carrier,lane,current_location,customer_tier,sku,supplier_id,reported_delay_hours,event_type,event_id,warehouse,days_of_cover,demand_signal
0,SO-104488,Contoso Freight,Newark Port to Chicago DC,New York City,platinum,MED-PUMP-42,SUP-8842,18.0,shipment_delay,37369999-a4c6-471f-aff3-b19a5b4ac8f4,NaN,NaN,NaN
1,NaN,NaN,NaN,Dallas,NaN,BATTERY-LFP-9K,NaN,NaN,inventory_risk,96c0a654-749d-4421-9ea5-58b9b4e0aa54,Dallas DC,2.4,promotion spike


## 7. Copilot Orchestration Function

This function is the thin Copilot layer around the Foundry Agent. It adds correlation IDs, packs the event as structured input, and calls the agent through the Responses API.


In [8]:
def invoke_supply_chain_copilot(event: dict[str, Any]) -> dict[str, Any]:
    conversation = openai_client.conversations.create()
    correlation_id = event["event_id"]

    prompt = f"""
Analyze this real-time supply-chain event. Use tools when useful, especially for weather or enterprise operations lookups.

Correlation ID: {correlation_id}
Event JSON:
{json.dumps(event, indent=2)}

Return a concise incident-response recommendation that an operations manager can act on.
""".strip()

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": agent_name, "type": "agent_reference"}},
        input=prompt,
    )

    return {"correlation_id": correlation_id, "conversation_id": conversation.id, "agent_response": response.output_text}


## 8. Run the Real-Time Demo

This cell processes the event stream one event at a time and prints the Copilot recommendation. In a live app, this loop becomes a consumer process and writes results to Teams, Dynamics, ServiceNow, Power BI, or an operations dashboard.


In [9]:
results = []

for event in realtime_events:
    console.rule(f"Processing {event['event_type']} - {event['event_id']}")
    result = invoke_supply_chain_copilot(event)
    results.append(result)
    console.print(Panel(result["agent_response"], title=f"Copilot Recommendation - {result['correlation_id']}"))
    time.sleep(1)

print(f"Processed {len(results)} event(s).")


──────────────────────── Processing shipment_delay - 37369999-a4c6-471f-aff3-b19a5b4ac8f4 ─────────────────────────

╭───────────────────────── Copilot Recommendation - 37369999-a4c6-471f-aff3-b19a5b4ac8f4 ─────────────────────────╮
│ 1. Executive summary                                                                                            │
│ Shipment SO-104488 (SKU MED-PUMP-42) for a platinum-tier customer is delayed 18 hours by Contoso Freight on the │
│ Newark Port–Chicago DC lane, currently at New York City as of 23 Jun 2026 20:34 UTC. Light rain/drizzle is      │
│ present but not severe enough to explain an 18-hour delay. Medical-pump item and premium customer elevate       │
│ response priority.                                                                                              │
│                                                                                                                 │
│ 2. Severity and confidence                                                                                      │
│ Severity: High (platinum customer + medical SKU risks downstream inventory/stockout at Chicago DC and potential │
│ SLA breach).                                                                                                    │
│ Confidence: 80 (event data direct; weather ruled out as primary cause).                                         │
│                                                                                                                 │
│ 3. Evidence used                                                                                                │
│ - Direct event payload: order_id SO-104488, carrier Contoso Freight, lane Newark–Chicago, location New York     │
│ City, reported_delay_hours 18, customer_tier platinum, SKU MED-PUMP-42.                                         │
│ - Real-time weather lookup (New York City, 23 Jun 2026): light rain/drizzle, 21–23 °C, 9–11 mph winds, no       │
│ storms or high-impact precipitation.                                                                            │
│ - Observed facts only; no inventory or ETA data yet.                                                            │
│                                                                                                                 │
│ 4. Recommended actions                                                                                          │
│ - Immediately request revised ETA and root-cause update from Contoso Freight operations (target within 30 min). │
│ - Check Chicago DC buffer stock for MED-PUMP-42 and trigger expedited replenishment if < 48 h coverage.         │
│ - Notify platinum customer proactively with expected new arrival window and mitigation option.                  │
│ - Evaluate rail/truck bypass or alternative carrier for partial load if delay exceeds 24 h.                     │
│                                                                                                                 │
│ 5. Human approval required                                                                                      │
│ - Approval to initiate customer notification and/or switch to backup carrier (Ops Manager sign-off needed       │
│ within 1 hour).                                                                                                 │
│                                                                                                                 │
│ 6. Follow-up questions                                                                                          │
│ - Any additional delay-cause details from Contoso (weather, port congestion, mechanical)?                       │
│ - Current inventory level and safety stock for MED-PUMP-42 at Chicago DC?                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────── Processing inventory_risk - 96c0a654-749d-4421-9ea5-58b9b4e0aa54 ─────────────────────────

╭───────────────────────── Copilot Recommendation - 96c0a654-749d-4421-9ea5-58b9b4e0aa54 ─────────────────────────╮
│ **1. Executive summary**                                                                                        │
│ Low inventory cover (2.4 days) for SKU BATTERY-LFP-9K at Dallas DC triggered by promotion-driven demand spike.  │
│ Immediate replenishment required to avoid stockout during peak sales period.                                    │
│                                                                                                                 │
│ **2. Severity and confidence**                                                                                  │
│ Severity: High (inventory below 3-day threshold during demand surge).                                           │
│ Confidence: 90% (direct telemetry from control-tower event; no conflicting signals observed).                   │
│                                                                                                                 │
│ **3. Evidence used**                                                                                            │
│ - Event payload: event_type=inventory_risk, sku=BATTERY-LFP-9K, warehouse=Dallas DC, days_of_cover=2.4,         │
│ demand_signal=promotion spike, occurred_at=2026-06-23T20:34:32Z.                                                │
│ - Correlation ID: 96c0a654-749d-4421-9ea5-58b9b4e0aa54.                                                         │
│ - No weather or supplier disruptions detected in available data for Dallas location.                            │
│                                                                                                                 │
│ **4. Recommended actions**                                                                                      │
│ - Expedite next replenishment PO for BATTERY-LFP-9K to Dallas DC with priority freight (target arrival <48      │
│ hours).                                                                                                         │
│ - Activate demand throttling or substitute SKUs on promotion landing pages and call center scripts.             │
│ - Increase safety stock target to 5+ days and configure auto-reorder alert at 3.0 days.                         │
│ - Notify sales and marketing teams to pause or limit further promotion exposure until cover recovers.           │
│                                                                                                                 │
│ **5. Human approval required**                                                                                  │
│ Yes – expedited freight surcharge and any change to promotion terms require Supply-Chain Director or Operations │
│ Manager sign-off within next 2 hours.                                                                           │
│                                                                                                                 │
│ **6. Follow-up questions**                                                                                      │
│ None at this time; ready for execution once approval received.                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Processed 2 event(s).


## 9. Governance and Audit Export

Enterprise projects should preserve the request envelope, agent response, correlation IDs, and human decision state. This simple dataframe is a starting point for downstream persistence.


In [10]:
audit_rows = [
    {"correlation_id": result["correlation_id"], "conversation_id": result["conversation_id"], "agent_name": agent_name, "review_status": "pending_human_review", "created_at": datetime.now(timezone.utc).isoformat(), "response_preview": result["agent_response"][:500]}
    for result in results
]

audit_df = pd.DataFrame(audit_rows)
audit_df


,correlation_id,conversation_id,agent_name,review_status,created_at,response_preview
0,37369999-a4c6-471f-aff3-b19a5b4ac8f4,conv_df40a920941da7fe00Wk1gPAqTOmOfkiAM2i3bBSQ...,ent-supply-chain-control-tower-agent,pending_human_review,2026-06-23T20:35:05.778370+00:00,1. Executive summary\nShipment SO-104488 (SKU ...
1,96c0a654-749d-4421-9ea5-58b9b4e0aa54,conv_17572c22a0fb12480015LJIQK9HOYKTBebnISuf1Z...,ent-supply-chain-control-tower-agent,pending_human_review,2026-06-23T20:35:05.778370+00:00,**1. Executive summary** \nLow inventory cove...


## Production Hardening Checklist

- Put enterprise APIs behind Azure API Management with Entra ID, mTLS, or managed identity where supported.
- Log correlation IDs to Application Insights or your SIEM.
- Add a human approval workflow for rerouting freight, changing suppliers, or customer-impacting commitments.
- Store agent outputs and source events in a governed data store with retention policies.
- Add evaluation datasets for high-severity incident handling, hallucination checks, and action-quality scoring.
